# Matched Filter for Gravitational Wave Detection

This notebook implements a **matched filter** for detecting gravitational wave signals from binary neutron star (BNS) mergers using the [ripple](https://github.com/tedwards2412/ripple) library.

## Background

Matched filtering is the optimal linear technique for detecting a known signal in stationary Gaussian noise. For gravitational wave data with noise power spectral density $S_n(f)$, the matched filter SNR is:

$$\rho(t_c) = \frac{\langle s | h \rangle}{\sqrt{\langle h | h \rangle}}$$

where the noise-weighted inner product is:

$$\langle a | b \rangle = 4 \, \mathrm{Re} \int_0^\infty \frac{\tilde{a}^*(f)\, \tilde{b}(f)}{S_n(f)} \, df$$

The SNR time series (scanning over coalescence time $t_c$) is computed efficiently via an inverse FFT:

$$z(t) = 4 \int_0^\infty \frac{\tilde{h}^*(f)\, \tilde{s}(f)}{S_n(f)} e^{2\pi i f t} \, df$$

$$\rho(t) = \frac{|z(t)|}{\sigma_h}, \qquad \sigma_h^2 = 4 \int_0^\infty \frac{|\tilde{h}(f)|^2}{S_n(f)} \, df$$

## Outline

1. **Waveform generation** — use `ripplegw` to generate a BNS template (TaylorF2)
2. **Noise model** — analytic aLIGO-like PSD
3. **Signal injection** — inject a waveform at a target SNR into coloured Gaussian noise
4. **Matched filter** — compute the SNR time series via IFFT
5. **Detection** — identify the peak and compare parameters

## 1. Imports

In [ ]:
%config InlineBackend.figure_format = 'retina'

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import tukey

import jax
import jax.numpy as jnp
from jax import config

# Enable 64-bit precision for accurate waveform generation
config.update("jax_enable_x64", True)

from ripplegw.waveforms import TaylorF2
from ripplegw import ms_to_Mc_eta

print(f"JAX devices: {jax.devices()}")

## 2. Noise PSD

We use a simple analytic approximation to the Advanced LIGO design sensitivity PSD (zero-detuned high-power configuration).

In [ ]:
def aLIGO_psd(f):
    """
    Analytic fit to the aLIGO zero-detuned high-power PSD.
    
    From Ajith & Bose (2009), parametrised as:
        S_n(f) = S_0 * [ (f/f0)^(-4.14) - 5*(f/f0)^(-2)
                          + 111*(1 - (f/f0)^2 + 0.5*(f/f0)^4) / (1 + 0.5*(f/f0)^2) ]
    Returns NaN (effectively infinite noise) below f_low.
    """
    f0 = 215.0          # Hz  (knee frequency)
    S0 = 1e-49          # Overall amplitude [Hz^-1]
    f_low = 10.0        # Low-frequency cutoff

    x = f / f0
    psd = S0 * (x**(-4.14) - 5.0 * x**(-2)
                + 111.0 * (1.0 - x**2 + 0.5 * x**4) / (1.0 + 0.5 * x**2))
    psd = np.where(f < f_low, np.inf, psd)
    psd = np.where(psd <= 0, np.inf, psd)   # Guard against negative values in tails
    return psd


# Plot the PSD
f_plot = np.geomspace(10, 2048, 5000)
psd_plot = aLIGO_psd(f_plot)

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(f_plot, np.sqrt(psd_plot), color='steelblue')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel(r'$\sqrt{S_n(f)}$ [Hz$^{-1/2}$]')
ax.set_title('aLIGO Design Sensitivity')
ax.set_xlim(10, 2048)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Generate BNS Template with ripple

We use the **TaylorF2** waveform, which is the standard frequency-domain post-Newtonian approximant for BNS systems. The signal parameters are specified as:

| Parameter | Symbol | Unit |
|-----------|--------|------|
| Chirp mass | $\mathcal{M}_c$ | $M_\odot$ |
| Symmetric mass ratio | $\eta$ | — |
| Spin 1 (z) | $\chi_1$ | — |
| Spin 2 (z) | $\chi_2$ | — |
| Luminosity distance | $d_L$ | Mpc |
| Coalescence time | $t_c$ | s |
| Coalescence phase | $\phi_c$ | rad |

In [ ]:
# ── Signal parameters ────────────────────────────────────────────────────────
m1_true = 1.4     # Solar masses
m2_true = 1.3     # Solar masses
chi1    = 0.02    # Aligned spin on body 1 (small for NS)
chi2    = 0.01    # Aligned spin on body 2
dist_mpc = 100.0  # Distance in Mpc
tc_true  = 0.0    # Coalescence time within the segment (seconds)
phic     = 0.0    # Coalescence phase (radians)
lambda1  = 300.0  # Tidal deformability body 1 (set 0 to ignore)
lambda2  = 300.0  # Tidal deformability body 2

Mc_true, eta_true = ms_to_Mc_eta(jnp.array([m1_true, m2_true]))
print(f"Chirp mass  Mc  = {float(Mc_true):.4f} M_sun")
print(f"Mass ratio  eta = {float(eta_true):.4f}")

# ── Frequency grid ───────────────────────────────────────────────────────────
f_low  = 20.0     # Hz  — low-frequency cutoff for the analysis
f_high = 1024.0   # Hz  — Nyquist for 2048 Hz sampling
T      = 128.0    # seconds — segment duration (long enough for BNS inspiral)
del_f  = 1.0 / T  # frequency resolution

# One-sided positive-frequency grid (DC excluded)
f_grid = np.arange(f_low, f_high, del_f)
f_grid_jax = jnp.array(f_grid)

# TaylorF2 theta vector: [Mc, eta, chi1, chi2, dist_mpc, tc, phic, lambda1, lambda2]
theta_signal = jnp.array([
    Mc_true, eta_true,
    chi1, chi2,
    dist_mpc,
    tc_true,
    phic,
    lambda1, lambda2
])

f_ref = f_low   # Reference frequency for the waveform

# Generate the + and × polarisations in the frequency domain
hp_signal, hc_signal = TaylorF2.gen_TaylorF2_hphc(f_grid_jax, theta_signal, f_ref)
hp_signal = np.array(hp_signal)
hc_signal = np.array(hc_signal)

print(f"\nWaveform length : {len(hp_signal)} frequency bins")
print(f"Frequency range : {f_grid[0]:.1f} – {f_grid[-1]:.1f} Hz")

In [ ]:
# Plot the waveform amplitude and phase
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].loglog(f_grid, np.abs(hp_signal), label=r'$|\tilde{h}_+(f)|$', color='steelblue')
axes[0].loglog(f_plot, np.sqrt(psd_plot) / dist_mpc * 100, 
               '--', color='tomato', alpha=0.7, label=r'$\sqrt{S_n(f)}$ (rescaled)')
axes[0].set_ylabel('Strain amplitude [Hz$^{-1}$]')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)
axes[0].set_title(f'BNS Waveform — $m_1={m1_true}\,M_\\odot$, $m_2={m2_true}\,M_\\odot$, $d_L={dist_mpc}$ Mpc')

axes[1].semilogx(f_grid, np.unwrap(np.angle(hp_signal)), color='steelblue')
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Phase [rad]')
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Inject Signal into Coloured Gaussian Noise

We generate a stretch of coloured Gaussian noise whose power spectrum matches the aLIGO PSD, then inject the signal at a chosen SNR.

In [ ]:
rng = np.random.default_rng(seed=42)

# ── Noise inner-product helper ────────────────────────────────────────────────
def inner_product(a_f, b_f, psd, df):
    """Noise-weighted inner product: 4 Re[ int a*(f) b(f) / Sn(f) df ]."""
    integrand = (np.conj(a_f) * b_f / psd).real
    return 4.0 * np.sum(integrand) * df


# ── Full frequency axis for the time-series (length = N = T * fs) ─────────────
fs_rate = 2 * f_high          # sampling rate [Hz]
N       = int(T * fs_rate)    # total number of samples
df      = 1.0 / T             # frequency resolution

# Positive-frequency bins aligned with np.fft.rfft output
f_rfft = np.fft.rfftfreq(N, d=1.0 / fs_rate)  # shape (N//2 + 1,)

psd_rfft = aLIGO_psd(f_rfft)   # Evaluate PSD on rfft grid
psd_rfft[0] = np.inf            # Zero-frequency (DC) — never used

# ── Generate coloured Gaussian noise ──────────────────────────────────────────
# Each frequency bin: variance = Sn(f) / (2 df)
sigma_noise = np.sqrt(psd_rfft / (2.0 * df))
sigma_noise[~np.isfinite(sigma_noise)] = 0.0   # DC + f < f_low bins

noise_f = (rng.standard_normal(len(f_rfft)) + 1j * rng.standard_normal(len(f_rfft))) \
          * sigma_noise / np.sqrt(2.0)
noise_t = np.fft.irfft(noise_f, n=N)           # Real-valued time series

# ── Place the signal on the analysis frequency grid ──────────────────────────
# Map hp_signal (defined on f_grid) onto f_rfft
h_rfft = np.zeros(len(f_rfft), dtype=complex)
# Find the indices in f_rfft that correspond to f_grid
idx_low  = int(np.round(f_low  / df))
idx_high = int(np.round(f_high / df))
n_bins   = len(hp_signal)

h_rfft[idx_low : idx_low + n_bins] = hp_signal

# ── Compute optimal SNR and rescale to target ─────────────────────────────────
snr_bins  = (f_rfft >= f_low) & (f_rfft < f_high) & np.isfinite(psd_rfft)
sigma_h   = np.sqrt(4.0 * np.sum(
    np.abs(h_rfft[snr_bins])**2 / psd_rfft[snr_bins]
) * df)

print(f"Optimal SNR before rescaling: {sigma_h:.2f}")

target_snr = 12.0    # Desired injection SNR
scale      = target_snr / sigma_h
h_rfft_inj = h_rfft * scale

# Verify
sigma_h_inj = np.sqrt(4.0 * np.sum(
    np.abs(h_rfft_inj[snr_bins])**2 / psd_rfft[snr_bins]
) * df)
print(f"Injection SNR after rescaling: {sigma_h_inj:.2f}  (target: {target_snr})")

# ── Combine signal + noise in time domain ─────────────────────────────────────
signal_t = np.fft.irfft(h_rfft_inj, n=N)
data_t   = signal_t + noise_t
t_axis   = np.arange(N) / fs_rate - T / 2.0   # Centre the segment at 0

In [ ]:
# Plot a snippet of the data around coalescence
window_s = 0.5   # seconds to display around tc
mask_t   = np.abs(t_axis - tc_true) < window_s

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_axis[mask_t], data_t[mask_t], lw=0.5, color='grey', alpha=0.7, label='Data (signal + noise)')
ax.plot(t_axis[mask_t], signal_t[mask_t], lw=1.2, color='steelblue', label='Injected signal')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Strain')
ax.set_title(f'BNS injection at SNR = {target_snr}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Matched Filter

The matched filter SNR time series is computed in the frequency domain using a single IFFT:

$$z(t) = 4 \int_0^\infty \frac{\tilde{h}^*(f)\, \tilde{s}(f)}{S_n(f)} e^{2\pi i f t} df \quad\Longrightarrow\quad \rho(t) = \frac{|z(t)|}{\sigma_h}$$

We use the **exact same template** as the injection to illustrate the optimal case, then show a slightly mismatched template.

In [ ]:
def matched_filter_snr(data_f, template_f, psd, df, N):
    """
    Compute the matched filter SNR time series.

    Parameters
    ----------
    data_f     : complex array, shape (N//2 + 1,)
                 One-sided frequency-domain data (rfft of time series).
    template_f : complex array, shape (N//2 + 1,)
                 Frequency-domain template (same frequency grid as data_f).
    psd        : real array, shape (N//2 + 1,)
                 One-sided PSD S_n(f).
    df         : float
                 Frequency resolution [Hz].
    N          : int
                 Length of the original time series.

    Returns
    -------
    snr_t  : real array, shape (N,)
             SNR time series |z(t)| / sigma_h.
    sigma_h : float
              Template normalisation (optimal SNR at the injected distance).
    """
    # Noise-weight: h*(f) * s(f) / Sn(f)
    safe_psd = np.where(np.isfinite(psd) & (psd > 0), psd, np.inf)
    integrand_f = np.conj(template_f) * data_f / safe_psd

    # Template normalisation  sigma_h^2 = 4 * int |h|^2 / Sn df
    sigma_h = np.sqrt(4.0 * np.sum(
        np.abs(template_f)**2 / safe_psd
    ).real * df)

    # z(t) = IFFT of 4 * h*(f) s(f) / Sn(f)  (with correct normalisation)
    # np.fft.irfft computes sum_k X_k e^{2pi i k n/N}  ->  multiply by N*df = fs_rate
    z_t = np.fft.irfft(4.0 * integrand_f, n=N) * (N * df)

    # irfft gives real output; SNR = |z(t)| / sigma_h
    # For a complex SNR (amplitude + phase) use:
    snr_t = np.abs(z_t) / sigma_h

    return snr_t, sigma_h


# ── FFT the data ──────────────────────────────────────────────────────────────
data_f = np.fft.rfft(data_t) / fs_rate   # Normalise to match the continuous FT convention

# ── Matched filter with the exact template ────────────────────────────────────
snr_t_exact, sigma_exact = matched_filter_snr(
    data_f, h_rfft_inj, psd_rfft, df, N
)

# Peak detection
peak_idx  = np.argmax(snr_t_exact)
peak_snr  = snr_t_exact[peak_idx]
peak_time = t_axis[peak_idx]

print(f"Peak SNR        : {peak_snr:.2f}")
print(f"Recovered time  : {peak_time:.4f} s  (true: {tc_true:.4f} s)")

In [ ]:
# Plot the SNR time series near coalescence
plot_window = 1.0   # seconds around true tc to display
mask_snr = np.abs(t_axis - tc_true) < plot_window

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_axis[mask_snr], snr_t_exact[mask_snr], lw=1.0, color='steelblue', label='Matched filter SNR')
ax.axvline(tc_true, color='tomato', ls='--', lw=1.5, label=f'True $t_c = {tc_true}$ s')
ax.axvline(peak_time, color='green', ls=':', lw=1.5, label=f'Recovered $t_c = {peak_time:.4f}$ s')
ax.axhline(target_snr, color='grey', ls='-.', alpha=0.6, label=f'Injection SNR = {target_snr}')
ax.set_xlabel('Time [s]')
ax.set_ylabel('SNR')
ax.set_title('Matched Filter SNR Time Series (exact template)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Template Bank Search

In practice the signal parameters are unknown. We evaluate the matched filter over a small grid of templates to mimic a template bank search.

For each template we record the maximum SNR within the search window and use it to estimate the parameters.

In [ ]:
# ── Template bank: grid over (m1, m2) ─────────────────────────────────────────
m1_grid = np.linspace(1.1, 1.8, 10)   # Solar masses
m2_grid = np.linspace(1.1, 1.7, 9)    # Solar masses  (enforce m2 <= m1)

bank_results = []   # (m1, m2, peak_snr, peak_time)

for m1_t in m1_grid:
    for m2_t in m2_grid:
        if m2_t > m1_t:
            continue

        Mc_t, eta_t = ms_to_Mc_eta(jnp.array([m1_t, m2_t]))
        theta_t = jnp.array([Mc_t, eta_t, chi1, chi2, dist_mpc,
                              tc_true, phic, lambda1, lambda2])

        hp_t, _ = TaylorF2.gen_TaylorF2_hphc(f_grid_jax, theta_t, f_ref)
        hp_t = np.array(hp_t)

        # Place template on rfft grid
        h_t_rfft = np.zeros(len(f_rfft), dtype=complex)
        h_t_rfft[idx_low : idx_low + n_bins] = hp_t

        snr_t_bank, _ = matched_filter_snr(data_f, h_t_rfft, psd_rfft, df, N)

        # Only search within ±0.5 s of expected coalescence
        search_mask = np.abs(t_axis - tc_true) < 0.5
        local_peak  = snr_t_bank[search_mask].max()
        local_t_idx = np.argmax(snr_t_bank[search_mask])
        local_time  = t_axis[search_mask][local_t_idx]

        bank_results.append((m1_t, m2_t, float(local_peak), float(local_time)))

bank_results = np.array(bank_results)   # columns: m1, m2, snr, t_peak
best_idx     = np.argmax(bank_results[:, 2])
best = bank_results[best_idx]

print(f"Best template  : m1 = {best[0]:.2f}, m2 = {best[1]:.2f} M_sun")
print(f"Best SNR       : {best[2]:.2f}")
print(f"Recovered time : {best[3]:.4f} s  (true: {tc_true:.4f} s)")
print(f"True params    : m1 = {m1_true:.2f}, m2 = {m2_true:.2f} M_sun")

In [ ]:
# ── Plot SNR as a function of template masses ─────────────────────────────────
m1_arr  = bank_results[:, 0]
m2_arr  = bank_results[:, 1]
snr_arr = bank_results[:, 2]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(m1_arr, m2_arr, c=snr_arr, cmap='viridis', s=80, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Peak SNR')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='tomato', zorder=5, label='True parameters')
ax.scatter(best[0], best[1], marker='D', s=120, color='lime', zorder=4, label='Best template')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Bank SNR Map')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Fitting Factor and Match

The **match** between two waveforms $h_1$ and $h_2$ is the maximised overlap:

$$\mathcal{M}(h_1, h_2) = \max_{t_c, \phi_c} \frac{\langle h_1 | h_2 \rangle}{\sqrt{\langle h_1 | h_1 \rangle \langle h_2 | h_2 \rangle}}$$

A match of 1 means the templates are identical. The **fitting factor** (FF) of the bank is the match between the signal and the best-fitting template.

In [ ]:
def compute_match(h1_f, h2_f, psd, df):
    """
    Compute the match (maximised over time and phase) between h1 and h2.
    Maximisation over t_c is done via IFFT; over phi_c via |z|.
    """
    safe_psd = np.where(np.isfinite(psd) & (psd > 0), psd, np.inf)

    norm1 = np.sqrt(4.0 * np.sum(np.abs(h1_f)**2 / safe_psd).real * df)
    norm2 = np.sqrt(4.0 * np.sum(np.abs(h2_f)**2 / safe_psd).real * df)

    integrand = np.conj(h1_f) * h2_f / safe_psd
    z = 4.0 * integrand * df   # Frequency-domain product (summed later)

    return 4.0 * np.abs(np.sum(z)).real / (norm1 * norm2)   # Phase-max overlap at t=0


def compute_match_max_time(h1_f, h2_f, psd, df, N):
    """
    Compute the match maximised over both time and phase via IFFT.
    """
    safe_psd = np.where(np.isfinite(psd) & (psd > 0), psd, np.inf)

    norm1 = np.sqrt(4.0 * np.sum(np.abs(h1_f)**2 / safe_psd).real * df)
    norm2 = np.sqrt(4.0 * np.sum(np.abs(h2_f)**2 / safe_psd).real * df)

    integrand_f = np.conj(h1_f) * h2_f / safe_psd
    z_t = np.fft.irfft(4.0 * integrand_f, n=N) * (N * df)

    return np.abs(z_t).max() / (norm1 * norm2)


# Compute match between the injected signal and each bank template
matches = []
for row in bank_results:
    m1_t, m2_t = row[0], row[1]
    Mc_t, eta_t = ms_to_Mc_eta(jnp.array([m1_t, m2_t]))
    theta_t = jnp.array([Mc_t, eta_t, chi1, chi2, dist_mpc, tc_true, phic, lambda1, lambda2])
    hp_t, _ = TaylorF2.gen_TaylorF2_hphc(f_grid_jax, theta_t, f_ref)
    hp_t = np.array(hp_t)

    h_t_rfft = np.zeros(len(f_rfft), dtype=complex)
    h_t_rfft[idx_low : idx_low + n_bins] = hp_t

    m = compute_match_max_time(h_rfft_inj, h_t_rfft, psd_rfft, df, N)
    matches.append(m)

matches    = np.array(matches)
best_match = matches.max()
best_m_idx = matches.argmax()

print(f"Fitting factor (best match) : {best_match:.4f}")
print(f"Best-match template         : m1 = {bank_results[best_m_idx, 0]:.2f}, "
      f"m2 = {bank_results[best_m_idx, 1]:.2f} M_sun")

In [ ]:
# ── Plot match values across the bank ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(bank_results[:, 0], bank_results[:, 1],
                c=matches, cmap='plasma', vmin=0.8, vmax=1.0, s=80, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=ax, label='Match')
ax.scatter(m1_true, m2_true, marker='*', s=300, color='cyan', zorder=5, label='True parameters')
ax.set_xlabel(r'$m_1$ [$M_\odot$]')
ax.set_ylabel(r'$m_2$ [$M_\odot$]')
ax.set_title('Template Match Map')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. JAX-accelerated Gradient of SNR

A key feature of `ripplegw` is that waveform generation is differentiable via JAX. This lets us compute the **gradient of the SNR** with respect to source parameters — useful for gradient-based template placement and Fisher matrix computation.

In [ ]:
# Convert PSD and data to JAX arrays on the analysis grid
psd_jax   = jnp.array(psd_rfft)
data_f_jax = jnp.array(data_f)
df_jax    = jnp.float64(df)

@jax.jit
def snr_at_tc(
    theta,              # [Mc, eta, chi1, chi2, dist_mpc, tc, phic, lam1, lam2]
    data_f_in,
    psd_in,
    f_grid_in,
    f_ref_in,
    idx_low_in,
    n_rfft,
    df_in,
):
    """
    Differentiable scalar SNR at tc=theta[5].
    Computes |<h(theta)|s>| / sigma_h.
    """
    hp, _ = TaylorF2.gen_TaylorF2_hphc(f_grid_in, theta, f_ref_in)

    # Zero-pad template onto the rfft grid
    h_padded = jnp.zeros(n_rfft, dtype=jnp.complex128)
    h_padded = h_padded.at[idx_low_in : idx_low_in + len(hp)].set(hp)

    safe_psd = jnp.where(jnp.isfinite(psd_in) & (psd_in > 0), psd_in, jnp.inf)

    # Inner product <h|s> at fixed tc (no IFFT — evaluate at tc=theta[5])
    phase_shift = jnp.exp(2j * jnp.pi * jnp.arange(n_rfft) * df_in * theta[5])
    integrand   = jnp.conj(h_padded) * data_f_in * phase_shift / safe_psd
    hs          = 4.0 * jnp.abs(jnp.sum(integrand)) * df_in

    sigma_h = jnp.sqrt(4.0 * jnp.sum(jnp.abs(h_padded)**2 / safe_psd).real * df_in)

    return hs / sigma_h


# Evaluate at true parameters
snr_val = snr_at_tc(
    theta_signal,
    data_f_jax, psd_jax, f_grid_jax,
    f_ref, idx_low, len(f_rfft), df_jax
)
print(f"Differentiable SNR at true params : {float(jnp.abs(snr_val)):.2f}")

# Gradient with respect to theta
grad_snr = jax.grad(lambda th: jnp.abs(
    snr_at_tc(th, data_f_jax, psd_jax, f_grid_jax, f_ref, idx_low, len(f_rfft), df_jax)
))

g = grad_snr(theta_signal)
param_names = ['Mc', 'eta', 'chi1', 'chi2', 'dist_mpc', 'tc', 'phic', 'lambda1', 'lambda2']
print("\nGradient of SNR w.r.t. source parameters:")
for name, gi in zip(param_names, g):
    print(f"  d(SNR)/d({name:10s}) = {float(gi):+.4e}")

## 9. Summary

| Step | Result |
|------|--------|
| Injection SNR | {target_snr} |
| Recovered SNR (exact template) | see peak above |
| Best bank template SNR | see bank search |
| Fitting factor | see match computation |

### Key takeaways

* **ripple** generates frequency-domain BNS waveforms (TaylorF2) that are fully differentiable via JAX.
* The matched filter is computed in $O(N \log N)$ time via an inverse FFT over the noise-weighted cross-correlation.
* The SNR peaks sharply at the true coalescence time, demonstrating the sensitivity of the matched filter.
* **Differentiability** of the SNR opens the door to gradient-based template bank placement (the core idea of SparseBank).